In [2]:
cd ~/Project5/edtech_churn_engine

/Users/abi/Project5/edtech_churn_engine


In [ ]:
import duckdb
import pandas as pd
import numpy as np
import scipy.stats as stats
# Recommended pattern: Context manager automatically closes connection
with duckdb.connect('edtech_churn_engine.duckdb') as con:
    df_preview = con.execute("SELECT * FROM stg_vle_interactions LIMIT 5").df()

# Connection is safely closed here

In [7]:
!kill -9 63124

In [8]:

# 1. Connect to DuckDB persistent database
con = duckdb.connect('edtech_churn_engine.duckdb')

# 2. Extract weekly clickstream activity per student (Weeks 1-12)
df_weekly_activity = con.execute("""
    SELECT 
        sv.student_id,
        reg.module_code,
        reg.presentation_code,
        reg.has_withdrawn,
        reg.is_early_withdrawal_14d,
        
        -- Map day offsets to 7-day course weeks (Day 0-6 = Week 1, Day 7-13 = Week 2, etc.)
        CASE 
            WHEN sv.interaction_day_offset < 0 THEN 0
            ELSE FLOOR(sv.interaction_day_offset / 7) + 1 
        END AS course_week,
        
        SUM(sv.daily_click_count) AS total_clicks,
        
        -- Estimate active learning minutes (Benchmark: ~1.5 mins per VLE click interaction)
        ROUND(SUM(sv.daily_click_count) * 1.5, 2) AS estimated_active_minutes
    FROM stg_vle_interactions sv
    INNER JOIN stg_student_registrations reg
        ON sv.student_id = reg.student_id
       AND sv.module_code = reg.module_code
       AND sv.presentation_code = reg.presentation_code
    WHERE sv.interaction_day_offset >= 0  -- Focus on active course duration
    GROUP BY 1, 2, 3, 4, 5, 6
""").df()

print("Weekly activity logs loaded! Total records:", len(df_weekly_activity))

Weekly activity logs loaded! Total records: 579438


In [9]:
# Filter activity for Weeks 1 through 12
df_active_12w = df_weekly_activity[
    (df_weekly_activity['course_week'] >= 1) & 
    (df_weekly_activity['course_week'] <= 12)
]

# Aggregate unique active students per presentation and course week
cohort_counts = df_active_12w.groupby(['presentation_code', 'course_week'])['student_id'].nunique().unstack()

# Calculate retention decay % relative to Week 1 initial active learners
week1_baseline = cohort_counts[1]
cohort_decay_matrix = cohort_counts.divide(week1_baseline, axis=0).round(4) * 100

print("--- 12-Week Student Active Engagement Decay Matrix (%) ---")
print(cohort_decay_matrix)

--- 12-Week Student Active Engagement Decay Matrix (%) ---
course_week         1.0    2.0     3.0    4.0    5.0    6.0    7.0    8.0   \
presentation_code                                                            
2013B              100.0  96.82  103.46  97.43  91.18  88.16  93.50  87.11   
2013J              100.0  98.55  100.61  93.79  92.88  87.39  91.54  87.82   
2014B              100.0  99.18   96.84  92.84  89.76  84.80  82.03  81.89   
2014J              100.0  97.07  101.61  92.85  91.96  86.10  81.51  83.15   

course_week         9.0    10.0   11.0   12.0  
presentation_code                              
2013B              76.02  75.80  75.91  78.78  
2013J              80.49  72.29  57.80  52.46  
2014B              75.69  75.37  70.37  71.23  
2014J              79.16  75.06  53.66  45.63  


In [10]:
# Pivot Week 1 and Week 2 engagement minutes per student
df_early_window = df_active_12w[df_active_12w['course_week'].isin([1, 2])].pivot_table(
    index=['student_id', 'module_code', 'presentation_code', 'has_withdrawn', 'is_early_withdrawal_14d'],
    columns='course_week',
    values='estimated_active_minutes',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Rename pivoted columns
df_early_window.rename(columns={1: 'week_1_minutes', 2: 'week_2_minutes'}, inplace=True)

# Bin Week 2 active learning minutes into strategic buckets
df_early_window['week_2_activity_bucket'] = pd.cut(
    df_early_window['week_2_minutes'],
    bins=[-0.01, 15, 45, 90, 180, np.inf],
    labels=['<15 mins (Inactive)', '15-44 mins (At-Risk)', '45-89 mins (Moderate)', '90-179 mins (Good)', '180+ mins (Optimal)']
)

# Calculate withdrawal rate per activity bucket
churn_threshold_summary = df_early_window.groupby('week_2_activity_bucket', observed=False).agg(
    total_students=('student_id', 'count'),
    withdrawn_students=('has_withdrawn', 'sum'),
    churn_probability_pct=('has_withdrawn', lambda x: round(x.mean() * 100, 2)),
    avg_week1_mins=('week_1_minutes', lambda x: round(x.mean(), 1))
).reset_index()

print("--- Week 2 Activity Buckets vs. Eventual Subscription Churn ---")
print(churn_threshold_summary)

--- Week 2 Activity Buckets vs. Eventual Subscription Churn ---
  week_2_activity_bucket  total_students  withdrawn_students  \
0    <15 mins (Inactive)            7322                2225   
1   15-44 mins (At-Risk)            5033                1140   
2  45-89 mins (Moderate)            4642                 966   
3     90-179 mins (Good)            4767                 866   
4    180+ mins (Optimal)            4661                 731   

   churn_probability_pct  avg_week1_mins  
0                  30.39            45.9  
1                  22.65            64.4  
2                  20.81            93.6  
3                  18.17           139.9  
4                  15.68           269.8  


In [11]:
# Separate active minutes for retained vs withdrawn students in Week 2
active_retained = df_early_window[~df_early_window['has_withdrawn']]['week_2_minutes']
active_withdrawn = df_early_window[df_early_window['has_withdrawn']]['week_2_minutes']

# Welch's t-test (unequal variances)
t_stat, p_val = stats.ttest_ind(active_retained, active_withdrawn, equal_var=False)

print("\n--- Welch's t-Test Results (Week 2 Active Minutes) ---")
print(f"t-Statistic: {t_stat:.4f}")
print(f"p-Value: {p_val:.4e}")

if p_val < 0.05:
    print("Conclusion: Reject H0 — Significant difference in Week 2 learning time between retained and churned students.")
else:
    print("Conclusion: Fail to reject H0 — No significant difference detected.")


--- Welch's t-Test Results (Week 2 Active Minutes) ---
t-Statistic: 17.7027
p-Value: 3.0358e-69
Conclusion: Reject H0 — Significant difference in Week 2 learning time between retained and churned students.


In [12]:
# Save flattened cohort decay table to DuckDB
df_flat_decay = cohort_decay_matrix.reset_index().melt(
    id_vars='presentation_code', 
    var_name='course_week', 
    value_name='retention_pct'
)
con.execute("CREATE OR REPLACE TABLE int_engagement_cohort_decay AS SELECT * FROM df_flat_decay")

# Save student early churn threshold metrics to DuckDB
con.execute("CREATE OR REPLACE TABLE int_student_churn_thresholds AS SELECT * FROM df_early_window")

con.close()
print("Phase 2 cohort and diagnostic matrices successfully saved to DuckDB!")

Phase 2 cohort and diagnostic matrices successfully saved to DuckDB!


In [14]:

con = duckdb.connect('edtech_churn_engine.duckdb')

# 1. Export 12-week retention cohort decay matrix
con.execute("COPY int_engagement_cohort_decay TO 'data/int_engagement_cohort_decay.csv' (HEADER, DELIMITER ',');")

# 2. Export student early churn thresholds & activity buckets
con.execute("COPY int_student_churn_thresholds TO 'data/int_student_churn_thresholds.csv' (HEADER, DELIMITER ',');")

# 3. Export assessment bottleneck & score performance staging
con.execute("COPY stg_assessment_results TO 'data/stg_assessment_results.csv' (HEADER, DELIMITER ',');")

# 4. Export student registration & withdrawal status
con.execute("COPY stg_student_registrations TO 'data/stg_student_registrations.csv' (HEADER, DELIMITER ',');")

con.close()
print("CSVs exported successfully! Ready for Tableau Public.")

CSVs exported successfully! Ready for Tableau Public.


In [16]:
#import duckdb

con = duckdb.connect('edtech_churn_engine.duckdb')

# Verify the exact student count and withdrawal rate per bucket
df_verify = con.execute("""
    SELECT 
        week_2_activity_bucket,
        COUNT(DISTINCT student_id) AS total_students_in_bucket,
        COUNT(DISTINCT CASE WHEN has_withdrawn THEN student_id END) AS withdrawn_students,
        ROUND(
            COUNT(DISTINCT CASE WHEN has_withdrawn THEN student_id END)::DOUBLE / 
            COUNT(DISTINCT student_id) * 100, 2
        ) AS true_churn_probability_pct
    FROM int_student_churn_thresholds
    GROUP BY week_2_activity_bucket
    ORDER BY true_churn_probability_pct DESC;
""").df()

print(df_verify)

  week_2_activity_bucket  total_students_in_bucket  withdrawn_students  \
0    <15 mins (Inactive)                      6992                2129   
1   15-44 mins (At-Risk)                      4939                1121   
2  45-89 mins (Moderate)                      4569                 955   
3     90-179 mins (Good)                      4671                 851   
4    180+ mins (Optimal)                      4481                 715   

   true_churn_probability_pct  
0                       30.45  
1                       22.70  
2                       20.90  
3                       18.22  
4                       15.96  
